In [37]:
import cv2
from pathlib import Path
import numpy as np

In [38]:
test_id_file = Path('/tf/01_code/mylittlecodes/SleepVST_baseline/data/kvss/A-test_set.txt')
video_dir = Path('/tf/00_data/#_2021_Sleep_Video/')

with open(test_id_file, 'r') as f:
    test_ids = f.read().splitlines()

In [39]:
test_ids = [id_.replace('.h5', '') for id_ in test_ids]

In [40]:
for test_id in sorted(test_ids):
    print(test_id)

A2019-EM-01-0021
A2019-EM-01-0040
A2019-EM-01-0063
A2019-EM-01-0070
A2019-EM-01-0074
A2019-EM-01-0097
A2019-EM-01-0102
A2019-EM-01-0107
A2019-EM-01-0111
A2019-EM-01-0158
A2019-EM-01-0168
A2019-EM-01-0189
A2019-EM-01-0192
A2019-EM-01-0207
A2019-EM-01-0213
A2019-EM-01-0217
A2019-EM-01-0219
A2019-EM-01-0224
A2019-EM-01-0244
A2020-EM-01-0001
A2020-EM-01-0020
A2020-EM-01-0025
A2020-EM-01-0027
A2020-EM-01-0038
A2020-EM-01-0046
A2020-EM-01-0060
A2020-EM-01-0061
A2020-EM-01-0081
A2020-EM-01-0086
A2020-EM-01-0090
A2020-EM-01-0110
A2020-EM-01-0142
A2020-EM-01-0168
A2020-EM-01-0172
A2020-EM-01-0184
A2020-EM-01-0187
A2020-EM-01-0189
A2020-EM-01-0197
A2020-EM-01-0206
A2020-EM-01-0210
A2020-EM-01-0213
A2020-EM-01-0216
A2020-EM-01-0232
A2020-EM-01-0237
A2020-EM-01-0240
A2020-EM-01-0245
A2020-EM-01-0251
A2020-EM-01-0252
A2020-EM-01-0266
A2020-EM-01-0275
A2021-EM-01-0025
A2021-EM-01-0027
A2021-EM-01-0028
A2021-EM-01-0041
A2021-EM-01-0047
A2021-EM-01-0056
A2021-EM-01-0082
A2021-EM-01-0084
A2021-EM-01-00

In [ ]:
for test_id in test_ids:
    video_path = video_dir / test_id / f'{test_id}_video_01.mp4'
    cap = cv2.VideoCapture(str(video_path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    frame_indices = np.linspace(0, total_frames - 1, num=10, dtype=int)
    saved = 0
    for idx in frame_indices[1:-1]:  # 첫 프레임과 마지막 프레임은 제외
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            save_path = Path('test_sample') / test_id / f'frame_{saved}.jpg'
            save_path.parent.mkdir(parents=True, exist_ok=True)
            cv2.imwrite(str(save_path), frame)
            saved += 1
    print(f'Processed video: {test_id}, saved {saved} frames.')
    cap.release()

Processed video: A2020-EM-01-0110, saved 10 frames.
Processed video: A2019-EM-01-0158, saved 10 frames.
Processed video: A2021-EM-01-0161, saved 10 frames.
Processed video: A2021-EM-01-0090, saved 10 frames.
Processed video: A2019-EM-01-0074, saved 10 frames.
Processed video: A2021-EM-01-0041, saved 10 frames.
Processed video: A2020-EM-01-0168, saved 10 frames.
Processed video: A2021-EM-01-0047, saved 10 frames.
Processed video: A2019-EM-01-0107, saved 10 frames.
Processed video: A2021-EM-01-0112, saved 10 frames.
Processed video: A2020-EM-01-0060, saved 10 frames.
Processed video: A2019-EM-01-0070, saved 10 frames.
Processed video: A2020-EM-01-0237, saved 10 frames.
Processed video: A2020-EM-01-0197, saved 10 frames.
Processed video: A2019-EM-01-0213, saved 10 frames.
Processed video: A2019-EM-01-0219, saved 10 frames.
Processed video: A2019-EM-01-0102, saved 10 frames.
Processed video: A2020-EM-01-0216, saved 10 frames.
Processed video: A2020-EM-01-0061, saved 10 frames.
Processed vi

In [42]:
import pandas as pd
from tqdm import tqdm

In [ ]:
def video_mean_intensity_uniform_sample(video_path: Path, num_samples: int = 300) -> float:
    cap = cv2.VideoCapture(str(video_path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        return np.nan

    # 샘플링할 프레임 인덱스를 균등 생성
    idxs = np.linspace(0, total_frames - 1, num_samples, dtype=int)

    frame_means = []
    for idx in tqdm(idxs):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))  # 랜덤 접근(코덱에 따라 비용이 있을 수 있음)
        ret, frame = cap.read()
        if not ret:
            continue
        # IR이라면 grayscale가 더 의미에 맞고, 계산도 더 단순
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        frame_means.append(float(gray.mean()))

    cap.release()
    return float(np.mean(frame_means)) if frame_means else np.nan

intensity_dict = {}
for test_id in sorted(test_ids):
    video_path = video_dir / test_id / f"{test_id}_video_01.mp4"
    intensity = video_mean_intensity_uniform_sample(video_path, num_samples=300)
    intensity_dict[test_id] = intensity
    print(f"Video: {test_id}, intensity: {intensity}")

intensity_df = pd.DataFrame.from_dict(intensity_dict, orient="index", columns=["intensity"])
intensity_df.to_csv("test_video_intensity.csv")

  0%|          | 0/300 [00:00<?, ?it/s]

100%|██████████| 300/300 [00:17<00:00, 16.79it/s]


Video: A2019-EM-01-0021, intensity: 148.3968328125


100%|██████████| 300/300 [00:13<00:00, 23.02it/s]


Video: A2019-EM-01-0040, intensity: 118.91374945746529


...

100%|██████████| 300/300 [00:18<00:00, 16.46it/s]


Video: A2019-EM-01-0158, intensity: 157.75760179036456


100%|██████████| 300/300 [00:18<00:00, 16.25it/s]


Video: A2019-EM-01-0168, intensity: 152.7438888888889


 81%|████████  | 243/300 [00:19<00:03, 18.54it/s]

In [33]:
intensity_df = pd.read_csv("intensity.csv")
df = pd.read_csv("test_samples.csv")
merged_df = df.merge(intensity_df, left_on="id", right_on="id", how="left")
merged_df.to_csv("test_samples_with_intensity.csv", index=False)

In [34]:
stats_df = pd.read_csv("subject_stats_stage2.csv")
merged_df = df.merge(stats_df, left_on="id", right_on="subject")

In [36]:
merged_df.to_csv("test_samples_with_intensity.csv", index=False)